# Qwen2.5-Coder LoRA/QLoRA feasibility test on ContextCRBench

This notebook downloads ContextCRBench and runs a small adapter fine-tuning experiment for hunk-level merge/quality classification. A dedicated configuration cell selects the 3B or 7B reviewer and standard LoRA or 4-bit QLoRA; the active selection is 7B QLoRA.

> **Research-scope warning:** the current proposal keeps Qwen reviewers frozen and trains a compact context selector. Fine-tuning Qwen is therefore an exploratory feasibility test, not the main proposed method. Do not mix these results with the frozen-reviewer context-selection comparison.

## Experiment contract

- Base model: selected in the **Model and run configuration** cell below (active: `Qwen/Qwen2.5-Coder-7B-Instruct`)
- Method: selectable 4-bit QLoRA or BF16 LoRA; only adapter weights train
- Dataset: official ContextCRBench release (public Google Drive file linked by the authors)
- Task: predict the benchmark's `merged` field using only pre-decision inputs
- Default run: 2,560 training examples, 640 validation examples, 200 optimizer steps
- Split unit: repository (`full_name`), with all rows from a PR assigned together
- Excluded prompt fields: `merged`, review comments/verdicts, issue/PR comments, and other outcome fields

### First run

1. Select the project's `.venv` Python 3.11 kernel.
2. Run the installation cell below once. It replaces PyPI's CPU-only torch package with the official CUDA 13.0 wheel.
3. **Restart the notebook kernel after installation.** Then continue from the imports cell. A loaded CPU torch module cannot become CUDA-enabled without a restart.

**Validity caveat:** ContextCRBench uses merge status as the quality-estimation target. It is a benchmark proxy, not verified defect ground truth. Before scaling this run, manually audit chronology and confirm that issue/PR text was available at the intended review point.

In [1]:
# Run once, then RESTART THE KERNEL before continuing.
# RTX 50-series support requires a CUDA-enabled wheel; PyPI's generic Windows wheel is CPU-only.
%pip install --upgrade "torch==2.14.0+cu130" --index-url https://download.pytorch.org/whl/cu130
%pip install -U "transformers>=4.56,<6" "accelerate>=1.5" "datasets>=3.4" \
    "peft>=0.15" "bitsandbytes>=0.45" "gdown>=5.2" "scikit-learn>=1.6" \
    "ipywidgets>=8.1"

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import hashlib
import json
import random
import re
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents) if (path / ".git").exists()),
    WORKING_DIR,
)
print(f"Project root: {PROJECT_ROOT}")

W0927 03:08:53.033000 31416 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Project root: F:\documents\workspace\CS536_Project


## Model and run configuration

- Change `MODEL_SIZE` to switch between `"3b"` and `"7b"`.
- Change `USE_QLORA` to select 4-bit QLoRA (`True`) or ordinary BF16 LoRA (`False`).
- Every model/method combination writes to a separate checkpoint directory.

After changing either setting, restart the kernel and rerun from the imports cell. Standard BF16 LoRA for 7B is expected to exceed a 16 GB GPU; use QLoRA for the 7B local run.

In [3]:
# ==================== EDIT THIS CELL TO CONFIGURE A RUN ====================
MODEL_SIZE = "7b"  # Supported values: "3b", "7b"
USE_QLORA = True   # True: 4-bit QLoRA; False: BF16 LoRA (7B likely OOM on 16 GB)

MODEL_PRESETS = {
    "3b": {"model_name": "Qwen/Qwen2.5-Coder-3B-Instruct"},
    "7b": {"model_name": "Qwen/Qwen2.5-Coder-7B-Instruct"},
}
if MODEL_SIZE not in MODEL_PRESETS:
    raise ValueError(f"MODEL_SIZE must be one of {sorted(MODEL_PRESETS)}, got {MODEL_SIZE!r}")

SEED = 42
MAX_TRAIN_EXAMPLES = 2_560
MAX_EVAL_EXAMPLES = 640
MAX_SEQ_LENGTH = 2048
MAX_STEPS = 200

preset = MODEL_PRESETS[MODEL_SIZE]
MODEL_NAME = preset["model_name"]
RUN_METHOD = "qlora" if USE_QLORA else "lora"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints" / f"qwen2.5-coder-{MODEL_SIZE}-contextcrbench-{RUN_METHOD}-smoke"
DATA_ROOT = PROJECT_ROOT / "data" / "contextcrbench"
DOWNLOAD_ROOT = DATA_ROOT / "download"
EXTRACT_ROOT = DATA_ROOT / "extracted"
FINAL_JSON_ROOT = EXTRACT_ROOT / "final_json"
ARCHIVE_PATH = DOWNLOAD_ROOT / "contextcrbench_data.zip"
GOOGLE_DRIVE_FILE_ID = "1pEvBisl0komPDG9Tb91B6L1SaR696Np2"
EXPECTED_RECORD_FILES = 68_938
FORCE_DOWNLOAD = False
FORCE_EXTRACT = False
DATA_FILE: Path | None = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Active model: {MODEL_NAME}")
print(f"Adapter method: {'4-bit QLoRA' if USE_QLORA else 'BF16 LoRA'}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Train/eval examples: {MAX_TRAIN_EXAMPLES}/{MAX_EVAL_EXAMPLES}")
print(f"Steps / max sequence length: {MAX_STEPS} / {MAX_SEQ_LENGTH}")
if MODEL_SIZE == "7b" and not USE_QLORA:
    print("WARNING: BF16 LoRA for 7B will probably exceed 16 GB VRAM.")

Active model: Qwen/Qwen2.5-Coder-7B-Instruct
Adapter method: 4-bit QLoRA
Output directory: F:\documents\workspace\CS536_Project\checkpoints\qwen2.5-coder-7b-contextcrbench-qlora-smoke
Train/eval examples: 2560/640
Steps / max sequence length: 200 / 2048


In [4]:
# QLoRA through bitsandbytes requires NVIDIA CUDA. Stop early on CPU/MPS.
print(f"Python executable: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime bundled with torch: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        f"CUDA is unavailable in this kernel (torch={torch.__version__}, "
        f"torch.version.cuda={torch.version.cuda}). Run the installation cell, "
        "restart the kernel, confirm the selected kernel is this project's .venv, "
        "and then rerun from the imports cell."
    )

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

if props.total_memory < 14 * 1024**3:
    raise RuntimeError("At least about 14 GiB VRAM is recommended for this configuration.")

Python executable: f:\documents\workspace\CS536_Project\.venv\Scripts\python.exe
PyTorch: 2.14.0+cu130
CUDA runtime bundled with torch: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.9 GiB
BF16 supported: True


In [5]:
# Download the authors' 1.56 GB processed-data release. This does not run until you execute the cell.
import gdown

DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if FORCE_DOWNLOAD or not ARCHIVE_PATH.exists():
    downloaded = gdown.download(
        id=GOOGLE_DRIVE_FILE_ID,
        output=str(ARCHIVE_PATH),
        quiet=False,
        resume=True,
    )
    if downloaded is None:
        raise RuntimeError("Google Drive download failed; verify that the public file is still accessible.")
else:
    print(f"Using existing archive: {ARCHIVE_PATH}")

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

archive_sha256 = sha256_file(ARCHIVE_PATH)
print(f"Archive size: {ARCHIVE_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"Archive SHA-256: {archive_sha256}")

existing_record_count = (
    sum(1 for _ in FINAL_JSON_ROOT.glob("*.json"))
    if FINAL_JSON_ROOT.is_dir()
    else 0
)
if existing_record_count == EXPECTED_RECORD_FILES and not FORCE_EXTRACT:
    print(f"Extraction skipped: all {existing_record_count:,} JSON records already exist.")
elif existing_record_count not in {0, EXPECTED_RECORD_FILES} and not FORCE_EXTRACT:
    raise RuntimeError(
        f"Found a partial extraction ({existing_record_count:,}/{EXPECTED_RECORD_FILES:,} records). "
        "Inspect it, then set FORCE_EXTRACT=True to extract again deliberately."
    )
else:
    print("Extracting dataset archive...")
    shutil.unpack_archive(ARCHIVE_PATH, EXTRACT_ROOT)
    extracted_record_count = sum(1 for _ in FINAL_JSON_ROOT.glob("*.json"))
    if extracted_record_count != EXPECTED_RECORD_FILES:
        raise RuntimeError(
            f"Extraction produced {extracted_record_count:,} records; expected {EXPECTED_RECORD_FILES:,}."
        )
    print(f"Extracted {extracted_record_count:,} records to: {FINAL_JSON_ROOT}")

Using existing archive: F:\documents\workspace\CS536_Project\data\contextcrbench\download\contextcrbench_data.zip
Archive size: 1.56 GiB
Archive SHA-256: 11eb643a9fb98d57044e0647cdcc9a5167ab16b39f8f2b84fb9fe394348fef37
Extraction skipped: all 68,938 JSON records already exist.


In [6]:
# The official release is one JSON object per record, not a JSONL table.
# Hugging Face's JSON loader cannot parse those nested objects, so index them directly.
from tqdm.auto import tqdm

REQUIRED_COLUMNS = {
    "full_name", "pr_number", "lang", "diff_hunk", "merged",
    "issue_body", "pr_title", "pr_body", "snippet_before", "snippet_after",
}
PROMPT_COLUMNS = [
    "full_name", "pr_number", "lang", "diff_hunk", "merged",
    "issue_body", "pr_title", "pr_body", "snippet_before", "snippet_after",
]

def load_record(path: Path) -> dict:
    with path.open(encoding="utf-8") as handle:
        record = json.load(handle)
    if not isinstance(record, dict):
        raise TypeError(f"{path} does not contain one JSON object")
    return record

record_paths = sorted(EXTRACT_ROOT.rglob("*.json"))
if not record_paths:
    raise FileNotFoundError(f"No per-record JSON files found under {EXTRACT_ROOT}")

first_record = load_record(record_paths[0])
missing_schema = REQUIRED_COLUMNS - set(first_record)
if missing_schema:
    raise KeyError(f"Missing required columns: {sorted(missing_schema)}")
ISSUE_TITLE_COLUMN = "issue_title" if "issue_title" in first_record else "issue_tilte"
if ISSUE_TITLE_COLUMN not in first_record:
    raise KeyError("Neither issue_title nor issue_tilte is present.")

audit_columns = [ISSUE_TITLE_COLUMN, "issue_body", "pr_title", "pr_body", "snippet_before", "snippet_after"]
missing_counts = dict.fromkeys(audit_columns, 0)
index_rows = []
for path in tqdm(record_paths, desc="Indexing ContextCRBench"):
    record = load_record(path)
    missing_record_columns = REQUIRED_COLUMNS - set(record)
    if missing_record_columns:
        raise KeyError(f"{path.name} is missing columns: {sorted(missing_record_columns)}")
    index_rows.append({
        "path": str(path),
        "full_name": record.get("full_name"),
        "pr_number": record.get("pr_number"),
        "lang": record.get("lang"),
        "merged": record.get("merged"),
    })
    for column in audit_columns:
        value = record.get(column)
        if value is None or str(value).strip() == "":
            missing_counts[column] += 1

DATA_FILE = record_paths[0].parent
print(f"Indexed {len(index_rows):,} records from {DATA_FILE}")

Indexing ContextCRBench:   0%|          | 0/68938 [00:00<?, ?it/s]

Indexed 68,938 records from F:\documents\workspace\CS536_Project\data\contextcrbench\extracted\final_json


In [7]:
# Schema checks already ran while indexing. This cell reports the resulting audit.
for row in index_rows[:5]:
    print({key: row[key] for key in ("full_name", "pr_number", "lang", "merged")})

label_counts = {}
for row in index_rows:
    label = str(row["merged"])
    label_counts[label] = label_counts.get(label, 0) + 1
print("\nLabel counts:")
print(label_counts)
print("\nMissing-value rates for model inputs:")
for column, count in missing_counts.items():
    print(f"{column:20s} {count / len(index_rows):.1%}")

{'full_name': 'bevyengine/bevy', 'pr_number': 6870, 'lang': 'rust', 'merged': False}
{'full_name': 'bevyengine/bevy', 'pr_number': 6870, 'lang': 'rust', 'merged': False}
{'full_name': 'bevyengine/bevy', 'pr_number': 2713, 'lang': 'rust', 'merged': False}
{'full_name': 'bevyengine/bevy', 'pr_number': 1525, 'lang': 'rust', 'merged': False}
{'full_name': 'bevyengine/bevy', 'pr_number': 2241, 'lang': 'rust', 'merged': False}

Label counts:
{'False': 13730, 'True': 55208}

Missing-value rates for model inputs:
issue_title          0.0%
issue_body           0.1%
pr_title             0.0%
pr_body              0.1%
snippet_before       12.5%
snippet_after        0.1%


In [8]:
# Repository-held-out split; rows from the same PR cannot cross partitions.
def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer)):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Unsupported merged label: {value!r}")

repositories = sorted({row["full_name"] for row in index_rows if row["full_name"]})
rng = random.Random(SEED)
rng.shuffle(repositories)
cut = max(1, int(0.8 * len(repositories)))
train_repos = set(repositories[:cut])
eval_repos = set(repositories[cut:])

train_pool = [index for index, row in enumerate(index_rows) if row["full_name"] in train_repos]
eval_pool = [index for index, row in enumerate(index_rows) if row["full_name"] in eval_repos]

def balanced_indices(pool: list[int], limit: int) -> list[int]:
    by_label = {False: [], True: []}
    for index in pool:
        by_label[as_bool(index_rows[index]["merged"])].append(index)
    per_class = limit // 2
    if min(map(len, by_label.values())) < per_class:
        raise ValueError(f"Not enough examples per class for requested limit={limit}")
    for indices in by_label.values():
        rng.shuffle(indices)
    chosen = by_label[False][:per_class] + by_label[True][:per_class]
    rng.shuffle(chosen)
    return chosen

def load_selected(indices: list[int]) -> Dataset:
    columns = PROMPT_COLUMNS + [ISSUE_TITLE_COLUMN]
    rows = []
    for index in indices:
        record = load_record(Path(index_rows[index]["path"]))
        rows.append({column: record.get(column) for column in columns})
    return Dataset.from_list(rows)

train_rows = balanced_indices(train_pool, MAX_TRAIN_EXAMPLES)
eval_rows = balanced_indices(eval_pool, MAX_EVAL_EXAMPLES)
train_raw = load_selected(train_rows)
eval_raw = load_selected(eval_rows)

train_prs = set(zip(train_raw["full_name"], train_raw["pr_number"]))
eval_prs = set(zip(eval_raw["full_name"], eval_raw["pr_number"]))
assert train_prs.isdisjoint(eval_prs)
assert set(train_raw["full_name"]).isdisjoint(set(eval_raw["full_name"]))
print(f"Train: {len(train_raw)} rows from {len(set(train_raw['full_name']))} repositories")
print(f"Eval:  {len(eval_raw)} rows from {len(set(eval_raw['full_name']))} repositories")

Train: 2560 rows from 73 repositories
Eval:  640 rows from 19 repositories


In [9]:
SYSTEM_PROMPT = """You are a code reviewer. Decide whether the supplied pull request change should be merged. Use only the supplied diff and context. Return exactly one JSON object with one Boolean field named is_merged; do not add markdown or explanation."""

# Character caps prevent one field from monopolizing the model window; tokenization applies the final cap.
FIELD_CHAR_LIMITS = {
    "diff_hunk": 12_000,
    "issue_body": 4_000,
    "pr_body": 4_000,
    "snippet_before": 8_000,
    "snippet_after": 8_000,
}

def clean_text(value, limit: int) -> str:
    if value is None:
        return ""
    return str(value).replace("\x00", "")[:limit].strip()

def build_user_prompt(example: dict) -> str:
    sections = [
        f"Repository: {clean_text(example['full_name'], 500)}",
        f"Language: {clean_text(example['lang'], 100)}",
        f"Diff hunk:\n{clean_text(example['diff_hunk'], FIELD_CHAR_LIMITS['diff_hunk'])}",
    ]
    optional = [
        ("Issue title", clean_text(example[ISSUE_TITLE_COLUMN], 500)),
        ("Issue body", clean_text(example["issue_body"], FIELD_CHAR_LIMITS["issue_body"])),
        ("PR title", clean_text(example["pr_title"], 500)),
        ("PR body", clean_text(example["pr_body"], FIELD_CHAR_LIMITS["pr_body"])),
        ("Code before", clean_text(example["snippet_before"], FIELD_CHAR_LIMITS["snippet_before"])),
        ("Code after", clean_text(example["snippet_after"], FIELD_CHAR_LIMITS["snippet_after"])),
    ]
    sections.extend(f"{name}:\n{text}" for name, text in optional if text)
    return "\n\n".join(sections)

def target_text(example: dict) -> str:
    return json.dumps({"is_merged": as_bool(example["merged"])}, separators=(",", ":"))

print(build_user_prompt(train_raw[0])[:2000])
print("\nTarget:", target_text(train_raw[0]))

Repository: netty/netty

Language: java

Diff hunk:
@@ -580,7 +581,7 @@ public static ByteBuf copiedBuffer(ByteBuffer... buffers) {
     public static ByteBuf copiedBuffer(CharSequence string, Charset charset) {
         ObjectUtil.checkNotNull(string, "string");
         if (CharsetUtil.UTF_8.equals(charset)) {
-            return copiedBufferUtf8(string);
+            return copiedBufferUtf8(string, 0, string.length());
         }
         if (CharsetUtil.US_ASCII.equals(charset)) {
             return copiedBufferAscii(string);

Issue title:
Improve performance of Unpooled.copiedBuffer from CharSequence and char[]

Issue body:
Hi, there. As we known `ByteBufUtil.wirteUtf8()` is faster than `ByteBufUtil.encodeString()` when the charset is `UTF-8`.
So, If we add some code in `Unpooled.copiedBuffer(CharSequence string, Charset charset)`  method to predicate the `charset` is `UTF-8` or not. If true call `ByteBufUtil.writeUtf8` method . This will improve the perf evident

Affected method

In [10]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model_kwargs = {
    "device_map": {"": 0},
    "dtype": compute_dtype,
}
if USE_QLORA:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
print(f"Loaded {MODEL_NAME} with {'4-bit QLoRA' if USE_QLORA else 'BF16 LoRA'}")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-Coder-7B-Instruct with 4-bit QLoRA
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [11]:
def tokenize_example(example: dict) -> dict[str, list[int]]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(example)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    answer = target_text(example) + tokenizer.eos_token
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    prompt_budget = MAX_SEQ_LENGTH - len(answer_ids)
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=prompt_budget,
    )["input_ids"]
    input_ids = prompt_ids + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

train_tokens = train_raw.map(tokenize_example, remove_columns=train_raw.column_names)
eval_tokens = eval_raw.map(tokenize_example, remove_columns=eval_raw.column_names)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

lengths = np.array([len(ids) for ids in train_tokens["input_ids"]])
print(f"Token lengths: min={lengths.min()}, median={np.median(lengths):.0f}, max={lengths.max()}")
print(f"At cap: {(lengths == MAX_SEQ_LENGTH).mean():.1%}")

Map:   0%|          | 0/2560 [00:00<?, ? examples/s]

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Token lengths: min=445, median=2048, max=2048
At cap: 66.5%


In [12]:
from tqdm.auto import tqdm

EVAL_GENERATION_EXAMPLES = min(320, len(eval_raw))

def parse_prediction(text: str) -> bool | None:
    match = re.search(r'"is_merged"\s*:\s*(true|false)', text, flags=re.IGNORECASE)
    return None if match is None else match.group(1).lower() == "true"

@torch.inference_mode()
def generate_prediction(example: dict) -> tuple[bool | None, str]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(example)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH - 32,
    ).to("cuda")
    output = model.generate(
        **batch,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(output[0, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_prediction(generated), generated

def evaluate_generation(dataset: Dataset, limit: int) -> dict[str, float | int]:
    model.eval()
    previous_cache_setting = model.config.use_cache
    model.config.use_cache = True
    labels, predictions = [], []
    malformed = 0
    for example in tqdm(dataset.select(range(limit))):
        prediction, _ = generate_prediction(example)
        labels.append(as_bool(example["merged"]))
        if prediction is None:
            malformed += 1
            predictions.append(False)  # Fixed conservative failure rule; do not drop failures.
        else:
            predictions.append(prediction)
    model.config.use_cache = previous_cache_setting
    return {
        "examples": limit,
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "malformed": malformed,
    }

# Run immediately before training to capture the base-model baseline.
baseline_metrics = evaluate_generation(eval_raw, EVAL_GENERATION_EXAMPLES)
baseline_metrics

  0%|          | 0/320 [00:00<?, ?it/s]

{'examples': 320,
 'accuracy': 0.515625,
 'macro_f1': 0.5039950399503995,
 'malformed': 200}

In [ ]:
from transformers import TrainerCallback

# These are teacher-forced completion diagnostics. They are useful during training,
# but the generated-label accuracy measured before/after training remains the primary metric.
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_completion_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    shifted_predictions = predictions[:, :-1]
    shifted_labels = labels[:, 1:]
    valid = shifted_labels != -100
    token_accuracy = float((shifted_predictions[valid] == shifted_labels[valid]).mean())
    exact_matches = []
    for row_predictions, row_labels, row_valid in zip(shifted_predictions, shifted_labels, valid):
        exact_matches.append(bool(np.all(row_predictions[row_valid] == row_labels[row_valid])))
    return {
        "completion_token_accuracy": token_accuracy,
        "completion_exact_match": float(np.mean(exact_matches)),
    }

class LocalMetricsCallback(TrainerCallback):
    """Print concise metrics and persist every Trainer log event as JSONL."""

    def __init__(self, path: Path) -> None:
        self.path = path
        self.history = []
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text("", encoding="utf-8")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        record = {"step": state.global_step, **logs}
        self.history.append(record)
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, default=float) + "\n")
        shown = ["loss", "eval_loss", "eval_completion_token_accuracy", "eval_completion_exact_match", "learning_rate", "grad_norm"]
        summary = [f"step={state.global_step}"]
        summary.extend(f"{key}={record[key]:.5g}" for key in shown if key in record)
        print(" | ".join(summary), flush=True)

metrics_callback = LocalMetricsCallback(OUTPUT_DIR / "training_log.jsonl")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=max(1, int(0.1 * MAX_STEPS)),
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=50,
    eval_accumulation_steps=1,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch_fused",
    report_to="none",
    disable_tqdm=False,
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=eval_tokens,
    data_collator=data_collator,
    compute_metrics=compute_completion_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[metrics_callback],
)

# This is the cell that actually fine-tunes the LoRA adapters.
train_result = trainer.train()
print("Final training metrics:")
print(json.dumps(train_result.metrics, indent=2, default=float))
print(f"Detailed log: {metrics_callback.path}")

Step,Training Loss,Validation Loss


step=1 | loss=4.6967 | learning_rate=0 | grad_norm=2.9639
step=2 | loss=3.2574 | learning_rate=1e-05 | grad_norm=2.5837
step=3 | loss=4.6203 | learning_rate=2e-05 | grad_norm=5.8323
step=4 | loss=4.26 | learning_rate=3e-05 | grad_norm=3.4464
step=5 | loss=3.0092 | learning_rate=4e-05 | grad_norm=2.967
step=6 | loss=3.5099 | learning_rate=5e-05 | grad_norm=3.4441
step=7 | loss=4.1812 | learning_rate=6e-05 | grad_norm=6.1653
step=8 | loss=4.3722 | learning_rate=7e-05 | grad_norm=3.3849
step=9 | loss=3.0957 | learning_rate=8e-05 | grad_norm=3.7077
step=10 | loss=3.8197 | learning_rate=9e-05 | grad_norm=3.6133


In [ ]:
# Evaluate the adapted model on exactly the same held-out examples, then save adapters only.
post_tune_metrics = evaluate_generation(eval_raw, EVAL_GENERATION_EXAMPLES)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(OUTPUT_DIR)

run_manifest = {
    "model": MODEL_NAME,
    "method": "4-bit QLoRA smoke test" if USE_QLORA else "BF16 LoRA smoke test",
    "use_qlora": USE_QLORA,
    "seed": SEED,
    "dataset_file": str(DATA_FILE),
    "dataset_archive_sha256": archive_sha256,
    "train_examples": len(train_raw),
    "eval_examples": len(eval_raw),
    "generation_eval_examples": EVAL_GENERATION_EXAMPLES,
    "max_steps": MAX_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_repositories": sorted(train_repos),
    "eval_repositories": sorted(eval_repos),
    "baseline": baseline_metrics,
    "post_tune": post_tune_metrics,
    "training_metrics": train_result.metrics,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, default=float),
    encoding="utf-8",
)

print(json.dumps({"baseline": baseline_metrics, "post_tune": post_tune_metrics}, indent=2))
print(f"Saved LoRA adapter and manifest to: {OUTPUT_DIR}")

## Interpretation and next gate

This smoke test establishes whether the selected Qwen adapters can be updated and whether the larger run moves held-out benchmark metrics. Two hundred steps on this subset are still not evidence of a useful model.

Before any scaled run:

1. Audit 100–200 examples for label meaning, chronology, duplicates, missing context, and leakage.
2. Freeze repository-level train/validation/test manifests on disk rather than recreating them interactively.
3. Compare both adapted models against their untouched frozen 3B and 7B baselines.
4. Keep the proposal's main experiment focused on training the compact context selector, not Qwen.
5. Report accuracy, macro-F1, malformed outputs, tokens, throughput, VRAM, and multiple seeds.